# 04 · Orchestrate — 01 Should I retrieve at all?

**Every retrieval notebook in this repo starts by retrieving. `04-retrieve/08-backend-cascade.ipynb` calls `retrieve()` the moment it has a query; `09-multi-query.ipynb` runs every phrasing before it has seen a single result. Nothing anywhere asks first whether a document lookup is needed. This notebook is that question, asked before anything else runs.**

The distinction this stage exists to land: a plain pipeline treats retrieval as
the workflow itself — step 1, always. An agent treats retrieval as one tool it
*chooses* whether to call. This notebook is the smallest possible version of
that choice, and deliberately the least clever one.

**Why the first version is rule-based, not a model.** A routing decision made
by a model has two ways to be wrong: the wiring can be broken, or the judgment
can be poor. Those look identical from the outside — a question that was
skipped when it should have been retrieved gives you no way to tell which
failed. A keyword/pattern rule has no judgment to blame, so the first version
proves the *wiring* works: the decision is read, acted on, and logged, and
retrieval genuinely does not happen when the decision says not to. A
model-driven router is layered on top only after that holds — and Step 7 shows
it never replaces the rule path, because the rule path is what this repo's
"run in 60 seconds, no key" promise depends on.

## What this notebook demonstrates

| Name | What it does | Example |
|---|---|---|
| `KNOWN_CONTEXT` | Facts the session already holds, so a lookup would be redundant | `KNOWN_CONTEXT["course_id"]` |
| `context_hit` | Checks a question against what is already known, before any rule fires | `context_hit("what course is this?")` → `"course_id"` |
| `RETRIEVAL_CUES` / `GENERAL_CUES` | The two pattern lists the decision is made from | `RETRIEVAL_CUES[0]` |
| `should_retrieve` | The whole decision: a verdict, the reason, and the rule that fired | `should_retrieve("according to the trial, ...")` → `{"retrieve": True, ...}` |
| `fixed_pipeline` | The always-retrieve baseline every other notebook in this repo implements | `fixed_pipeline(questions)` |
| `routed_pipeline` | The same questions, with the decision actually wired to the retrieval call | `routed_pipeline(questions)` |
| `model_route` | Optional model-driven verdict; falls back to the rule verdict with no key | `model_route(q, meter)` |

In [ ]:
import sys
from pathlib import Path

_root = Path.cwd().resolve()
for _ in range(6):
    if (_root / "nbio.py").is_file():
        break
    _root = _root.parent
else:
    raise RuntimeError("could not locate nbio.py above the current directory")
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import nbio
nbio.bootstrap()

## Step 1 — the baseline: a retriever that is never asked whether to run

`search_local` below is the same hash-embedded, in-memory store
`04-retrieve/08-backend-cascade.ipynb` uses, kept self-contained here as
every notebook in this repo is. `RETRIEVAL_CALLS` counts how many times it
actually executes, because "the decision was wired up" and "the decision
was printed and then ignored" produce identical-looking logs otherwise. A
call counter is the only thing that tells them apart.

In [ ]:
import hashlib
import math

RETRIEVAL_CALLS = []  # one entry per real store lookup, so the wiring is auditable


def hash_embed(text: str, dim: int = 384) -> list[float]:
    vec = [0.0] * dim
    tokens = (text or "").lower().split()
    if not tokens:
        return vec
    for tok in tokens:
        h = int(hashlib.sha256(tok.encode("utf-8")).hexdigest(), 16)
        idx = h % dim
        sign = 1.0 if (h >> 8) & 1 else -1.0
        vec[idx] += sign
    norm = math.sqrt(sum(v * v for v in vec)) or 1.0
    return [v / norm for v in vec]


def cosine(a: list[float], b: list[float]) -> float:
    dot = sum(x * y for x, y in zip(a, b))
    na = math.sqrt(sum(x * x for x in a)) or 1.0
    nb = math.sqrt(sum(x * x for x in b)) or 1.0
    return dot / (na * nb)


COURSE_STORE = [
    {"chunk_id": "bio201::c0", "text": "Mitochondria perform oxidative phosphorylation, converting nutrients into ATP."},
    {"chunk_id": "bio201::c1", "text": "Photosynthesis converts light energy into chemical energy stored in glucose."},
]
for _ch in COURSE_STORE:
    _ch["embedding"] = hash_embed(_ch["text"])


def search_local(query: str, top_k: int = 2) -> list[dict]:
    RETRIEVAL_CALLS.append(query)
    qvec = hash_embed(query)
    scored = [{"chunk_id": c["chunk_id"], "text": c["text"], "score": cosine(qvec, c["embedding"])}
              for c in COURSE_STORE]
    scored.sort(key=lambda x: -x["score"])
    return scored[:top_k]


print(f"store seeded with {len(COURSE_STORE)} chunks")
print("one probe:", f"{search_local('what do mitochondria do?')[0]['score']:.3f}")
print("retrieval calls so far:", len(RETRIEVAL_CALLS))

## Step 2 — what the session already knows

The cheapest skip of all: the answer is already in hand. A session that has
already been told its course id does not need a document lookup to repeat
it. This check runs *before* any keyword rule, because a cue word in a
question ("what does the syllabus say my course id is?") would otherwise
send a known fact off to the store.

The match is a plain keyword-overlap test, not semantics — that is the
whole point of starting here. It is easy to read, impossible to blame on
judgment, and it fails in exactly one obvious direction (an unusual
phrasing misses, and the question falls through to retrieval).

In [ ]:
KNOWN_CONTEXT = {
    "course_id": {"value": "bio201", "triggers": {"course", "id", "which class", "what class"}},
    "session_owner": {"value": "a teaching-assistant demo session", "triggers": {"who am i talking to", "session"}},
}


def context_hit(question: str) -> str | None:
    """The key of an already-known fact this question is asking for, or None."""
    q = (question or "").lower()
    for key, entry in KNOWN_CONTEXT.items():
        matched = [t for t in entry["triggers"] if t in q]
        # Two independent triggers, so a single common word ("id", "session")
        # inside an unrelated question does not claim the whole question.
        if len(matched) >= 2:
            return key
    return None


print("'what course id is this session?' ->", context_hit("what course id is this session?"))
print("'what does the paper say about ATP?' ->", context_hit("what does the paper say about ATP?"))

assert context_hit("what course id is this session?") == "course_id"
assert context_hit("what does the paper say about ATP?") is None

## Step 3 — the two pattern lists the verdict is made from

`RETRIEVAL_CUES` are the marks of a question that is *about a document*:
it names a source, asks for a citation, points at a corpus, or asks for
something time-bound that a static model cannot be trusted on.
`GENERAL_CUES` are the marks of a question no corpus can help with:
greetings, arithmetic, and questions about the assistant itself.

Both are deliberately small and hand-written. A cue list is not a claim
about language in general — it is a claim about *these* patterns, which is
the only kind of claim a keyword rule can honestly make.

In [ ]:
import re

RETRIEVAL_CUES = [
    (re.compile(r"\baccording to\b"), "attributes the answer to a source"),
    (re.compile(r"\b(paper|study|trial|guideline|syllabus|handbook|lecture|module|document)s?\b"), "names a document type"),
    (re.compile(r"\b(cite|citation|reference|source)s?\b"), "asks for a citation"),
    (re.compile(r"\b(pmid|pmc\d|doi)\b"), "contains an identifier that must be looked up"),
    (re.compile(r"\b(latest|recent|current|as of|20\d\d)\b"), "time-bound, so static knowledge is not trustworthy"),
    (re.compile(r"\b(evidence|cohort|meta-analysis|systematic review|randomi[sz]ed)\b"), "asks for evidence, not recall"),
    (re.compile(r"\bin (this|the) (course|corpus|collection)\b"), "scoped to a specific corpus"),
]

GENERAL_CUES = [
    (re.compile(r"^\s*(hi|hey|hello|thanks|thank you|good morning)\b"), "conversational, not a question about a document"),
    (re.compile(r"\b\d+\s*[-+*/x]\s*\d+\b"), "arithmetic, computable without any corpus"),
    (re.compile(r"\b(who are you|what can you do|how do you work)\b"), "about the assistant itself"),
    (re.compile(r"^\s*(summari[sz]e|rephrase|shorten|translate) (that|this|the above)\b"), "operates on text already in hand"),
]

print(f"{len(RETRIEVAL_CUES)} retrieval cues, {len(GENERAL_CUES)} general-knowledge cues")

## Step 4 — `should_retrieve`: the verdict, the reason, and the rule that fired

Three things come back, not one. A bare `True`/`False` is unauditable —
when a routing decision is wrong, the only useful question is *which rule
fired*, and a boolean cannot answer it. The returned record is what makes
the rest of this stage debuggable.

**The default when nothing matches is `retrieve`.** The two errors are not
symmetric: a needless lookup costs a few milliseconds against a local
store, while a skipped lookup produces a confident answer with no evidence
behind it — the exact failure `01-tools/05-gate` exists to catch. When the
rules have nothing to say, erring toward the cheap mistake is the
defensible default.

In [ ]:
def should_retrieve(question: str) -> dict:
    """A retrieval verdict with the reason and the rule that produced it."""
    q = (question or "").strip()
    if not q:
        return {"retrieve": False, "rule": "empty", "reason": "no question to answer"}

    key = context_hit(q)
    if key:
        return {"retrieve": False, "rule": "known-context",
                "reason": f"already known: {key} = {KNOWN_CONTEXT[key]['value']!r}"}

    for pattern, why in RETRIEVAL_CUES:
        if pattern.search(q.lower()):
            return {"retrieve": True, "rule": f"cue:{pattern.pattern}", "reason": why}

    for pattern, why in GENERAL_CUES:
        if pattern.search(q.lower()):
            return {"retrieve": False, "rule": f"general:{pattern.pattern}", "reason": why}

    return {"retrieve": True, "rule": "default",
            "reason": "no rule matched; defaulting to retrieval, the cheaper mistake"}


for probe in ["According to the trial, what was the primary endpoint?",
              "hello, can you help me?",
              "what course id is this session?",
              "what is 12 * 8?"]:
    v = should_retrieve(probe)
    print(f"  retrieve={str(v['retrieve']):<5} {probe!r}\n        -> {v['reason']}")

assert should_retrieve("According to the trial, what was the primary endpoint?")["retrieve"] is True
assert should_retrieve("hello, can you help me?")["retrieve"] is False
assert should_retrieve("what course id is this session?")["rule"] == "known-context"

## Step 5 — the labelled set, including the case the rules get wrong

Eight questions with the verdict a careful human would give. Seven are
what the cue lists were built for. The eighth — "What is the capital of
France?" — is general knowledge that the rules send to retrieval anyway,
because no cue matches and the default is to retrieve.

That miss is asserted here, not hidden. A keyword rule has no way to know
which facts a model already holds; it can only recognise the patterns it
was given. Naming the failure in an assert means it cannot silently change
without this notebook going red — which is the only useful thing a test of
a known limitation can do.

In [ ]:
LABELLED = [
    ("What does the 2024 sepsis guideline say about fluid resuscitation?", True),
    ("According to the retrieved trial, what was the primary endpoint?", True),
    ("Summarize the mitochondria material in this course.", True),
    ("Is there evidence from a randomized study for early excision?", True),
    ("hello, can you help me?", False),
    ("What is 12 * 8?", False),
    ("who are you?", False),
    ("What is the capital of France?", False),  # the known miss, below
]

rows, agreed = [], 0
for q, expected in LABELLED:
    v = should_retrieve(q)
    ok = v["retrieve"] == expected
    agreed += ok
    rows.append((q[:46], expected, v["retrieve"], "yes" if ok else "NO", v["rule"][:30]))

nbio.table(rows, ("question", "human", "rule", "agree", "rule that fired"))
print(f"\nagreement: {agreed}/{len(LABELLED)}")

known_miss = "What is the capital of France?"
assert should_retrieve(known_miss)["retrieve"] is True, "the known miss"
assert should_retrieve(known_miss)["rule"] == "default", "and it misses via the default, not a bad cue"
assert agreed == len(LABELLED) - 1, "exactly one labelled disagreement, the documented one"
print("confirmed: 7/8 match, and the 8th fails through the default branch, as documented")

## Step 6 — proving the wiring, not the judgment

This is the step the whole notebook is for. `fixed_pipeline` is what every
other retrieval notebook in this repo does: call the store, every time.
`routed_pipeline` reads the verdict and calls the store only when the
verdict says to.

The counter from Step 1 is the proof. If the decision were computed,
printed, and then ignored — the most common way a "routing layer" is
quietly broken — both counts would be identical.

In [ ]:
def fixed_pipeline(questions: list[str]) -> list[dict]:
    """Retrieval as the workflow: step 1, unconditionally."""
    return [{"question": q, "results": search_local(q)} for q in questions]


def routed_pipeline(questions: list[str]) -> list[dict]:
    """Retrieval as a tool the decision layer chooses to call, or not."""
    out = []
    for q in questions:
        verdict = should_retrieve(q)
        results = search_local(q) if verdict["retrieve"] else []
        out.append({"question": q, "verdict": verdict, "results": results})
    return out


questions = [q for q, _ in LABELLED]

RETRIEVAL_CALLS.clear()
fixed_pipeline(questions)
fixed_calls = len(RETRIEVAL_CALLS)

RETRIEVAL_CALLS.clear()
routed = routed_pipeline(questions)
routed_calls = len(RETRIEVAL_CALLS)

skipped = [r["question"] for r in routed if not r["verdict"]["retrieve"]]
print(f"fixed pipeline : {fixed_calls} store lookups for {len(questions)} questions")
print(f"routed pipeline: {routed_calls} store lookups for {len(questions)} questions")
print("\nskipped without a lookup:")
for s in skipped:
    print("  -", s)

assert fixed_calls == len(questions), "the baseline retrieves for every question by construction"
assert routed_calls < fixed_calls, "the verdict must actually suppress lookups, not just be printed"
assert routed_calls == len(questions) - len(skipped)
assert all(r["results"] == [] for r in routed if not r["verdict"]["retrieve"]), (
    "a skipped question must come back with no results at all"
)
print(f"\nconfirmed: {fixed_calls - routed_calls} lookups genuinely did not happen")

## Step 7 — the model-driven version, layered on, never replacing

A model can route questions the cue lists have never seen — including the
capital-of-France miss from Step 5. What it cannot do is run with no key,
which is the condition every notebook in this repo has to meet.

So the model is a *layer*: if a key is present, the model's verdict is
taken, under a spend ceiling, with the rule verdict kept alongside for
comparison. If no key is present, the function says so and returns the
rule verdict unchanged. Either way `should_retrieve` is still what decides
when nothing else can — deleting the rule path to "upgrade" to a model
would break the offline path this repo promises.

A fully general LLM-driven classifier — prompt design, a labelled
evaluation set, calibration of its confidence — is follow-on work, not
built here. This is the seam it would plug into.

In [ ]:
import os

ROUTER_PROMPT = (
    "Decide whether answering this question requires looking up documents in a "
    "corpus, or whether general knowledge is enough.\n"
    "Question: {q}\n"
    'Respond with JSON only: {{"retrieve": <true|false>, "why": "<one short clause>"}}'
)


def model_router_client():
    """(provider, model_id, client) if a key is set, else None. Never raises."""
    if os.environ.get("GROQ_API_KEY"):
        from groq import Groq
        return "groq", "llama-3.1-8b-instant", Groq(api_key=os.environ["GROQ_API_KEY"])
    if os.environ.get("OPENAI_API_KEY"):
        from openai import OpenAI
        return "openai", "gpt-4o-mini", OpenAI(api_key=os.environ["OPENAI_API_KEY"])
    return None


def model_route(question: str, meter=None) -> dict:
    """The model's verdict if a key is set; otherwise the rule verdict, unchanged."""
    rule_verdict = should_retrieve(question)
    client_info = model_router_client()
    if client_info is None:
        return {**rule_verdict, "source": "rule", "rule_verdict": rule_verdict["retrieve"]}

    provider, model_id, client = client_info
    try:
        resp = client.chat.completions.create(
            model=model_id,
            messages=[{"role": "user", "content": ROUTER_PROMPT.format(q=question)}],
            temperature=0.0, max_tokens=80,
        )
        if meter is not None:
            usage = getattr(resp, "usage", None)
            meter.record(model_id,
                         getattr(usage, "prompt_tokens", 0) if usage else 0,
                         getattr(usage, "completion_tokens", 0) if usage else 0)
        import json as jsonlib
        raw = resp.choices[0].message.content or ""
        start, end = raw.find("{"), raw.rfind("}")
        parsed = jsonlib.loads(raw[start:end + 1]) if start != -1 and end > start else {}
        if "retrieve" not in parsed:
            raise ValueError("no usable verdict in the model's reply")
        return {"retrieve": bool(parsed["retrieve"]), "rule": f"model:{model_id}",
                "reason": str(parsed.get("why", "")), "source": "model",
                "rule_verdict": rule_verdict["retrieve"]}
    except Exception as exc:
        print(f"model router unavailable ({type(exc).__name__}: {exc}) — using the rule verdict")
        return {**rule_verdict, "source": "rule", "rule_verdict": rule_verdict["retrieve"]}


if model_router_client() is None:
    print("GROQ_API_KEY / OPENAI_API_KEY not set, skipping the live router — "
          "deterministic stand-in instead: the rule verdict is used verbatim.\n")

with nbio.cost_meter(budget_usd=0.50) as meter:
    routed_verdicts = [model_route(q, meter=meter) for q, _ in LABELLED]
    print(meter.report())

sources = nbio.counts(routed_verdicts, "source")
print("\nverdict sources:", sources)

assert all(v["source"] in ("rule", "model") for v in routed_verdicts)
assert all(v["rule_verdict"] == should_retrieve(q)["retrieve"] for v, (q, _) in zip(routed_verdicts, LABELLED)), (
    "the rule verdict must survive intact alongside the model's, on every path"
)
print("confirmed: the rule verdict is computed and carried on every path, key or no key")

## What did not come across

- **A general router.** The cue lists recognise the patterns they were
  given and nothing else. Step 5's capital-of-France miss is the shape of
  every failure this design has: no cue matched, so it retrieved. Widening
  coverage means either more hand-written cues or the model layer in
  Step 7 — and the model layer is a seam, not a finished classifier. No
  prompt tuning, no labelled evaluation set beyond the eight questions
  above, no confidence calibration.
- **Eight questions is not an evaluation.** 7/8 agreement on a set chosen
  while writing the rules says the rules are self-consistent. It says
  nothing about held-out questions. Measuring that needs
  `04-benchmarks/clinical-retrieval/`, which is separate work.
- **Cost of a wrong skip.** This notebook counts lookups avoided. It does
  not measure what a wrongly-skipped lookup did to answer quality — that
  needs the grounding check in `01-tools/05-gate`, which
  `03-retry-on-verdict.ipynb` wires in next.